# Session 4 · Creating Audio and Video with Generative AI

Generative AI › Track 3 · Innovate with Generative AI

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-04_script_timer.py`

In [1]:
"""
Session 4 · Example 1 — How long will my narration be?

Text-to-speech and human narrators speak at roughly 130-160 words per minute
(wpm). Before generating any audio, time the script so each scene fits its
video clip. 150 wpm = 2.5 words per second.

Run:  python session-04_script_timer.py
"""
WPM = 150
script = [
    ("Hook",     "Ever wished your notes could talk back? Meet StudyBuddy."),
    ("Problem",  "Exams pile up, notes are messy, and nobody has time to make flashcards."),
    ("Solution", "StudyBuddy turns your own notes into quizzes and flashcards in seconds, "
                 "then reminds you to review just before you forget."),
    ("Proof",    "In our pilot, forty students used it for two weeks."),
    ("CTA",      "Try it free this week. Link in bio."),
]

total = 0.0
print(f"{'Scene':9} {'Words':>5} {'Seconds':>8}  Start-End")
for name, text in script:
    words = len(text.split())
    secs = words / WPM * 60
    print(f"{name:9} {words:5} {secs:8.1f}  {total:5.1f}-{total + secs:5.1f}s")
    total += secs
print(f"Total: {total:.1f} s at {WPM} wpm")
limit = 30
print("Fits a 30-second reel" if total <= limit else f"Too long by {total - limit:.1f} s: cut words")

Scene     Words  Seconds  Start-End
Hook          9      3.6    0.0-  3.6s
Problem      13      5.2    3.6-  8.8s
Solution     20      8.0    8.8- 16.8s
Proof        10      4.0   16.8- 20.8s
CTA           8      3.2   20.8- 24.0s
Total: 24.0 s at 150 wpm
Fits a 30-second reel


### Example · `session-04_srt_maker.py`

In [2]:
"""
Session 4 · Example 2 — Generate a subtitle (.srt) file from a timed script.

Captions matter: most social video is watched with the sound off, and
captions make content accessible to deaf and hard-of-hearing viewers.
An SRT file is plain text: a number, a time range "HH:MM:SS,mmm --> ...",
the caption text, then a blank line. Video editors (CapCut, Premiere,
DaVinci Resolve) and YouTube all import it.

Long lines are split so that no caption exceeds 42 characters per line,
which is a common broadcast guideline.

Run:  python session-04_srt_maker.py
Creates: session-04_promo.srt
"""
import textwrap

WPM = 150
MAX_CHARS = 42


def stamp(seconds: float) -> str:
    ms = round(seconds * 1000)
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"


def to_srt(lines):
    blocks, t, n = [], 0.0, 1
    for text in lines:
        # wrap into lines of at most MAX_CHARS, then show 2 lines per caption
        rows = textwrap.wrap(text, MAX_CHARS)
        for k in range(0, len(rows), 2):
            pair = rows[k:k + 2]
            words = sum(len(r.split()) for r in pair)
            dur = max(words / WPM * 60, 1.0)  # at least 1 second on screen
            blocks.append(f"{n}\n{stamp(t)} --> {stamp(t + dur)}\n" + "\n".join(pair) + "\n")
            t += dur
            n += 1
    return "\n".join(blocks)


script = [
    "Ever wished your notes could talk back? Meet StudyBuddy.",
    "Exams pile up, notes are messy, and nobody has time to make flashcards.",
    "StudyBuddy turns your own notes into quizzes and flashcards in seconds, "
    "then reminds you to review just before you forget.",
    "Try it free this week. Link in bio.",
]
srt = to_srt(script)
with open("session-04_promo.srt", "w", encoding="utf-8") as f:
    f.write(srt)
print(srt)
print("Saved -> session-04_promo.srt")

1
00:00:00,000 --> 00:00:03,600
Ever wished your notes could talk back?
Meet StudyBuddy.

2
00:00:03,600 --> 00:00:08,800
Exams pile up, notes are messy, and nobody
has time to make flashcards.

3
00:00:08,800 --> 00:00:13,600
StudyBuddy turns your own notes into
quizzes and flashcards in seconds, then

4
00:00:13,600 --> 00:00:16,800
reminds you to review just before you
forget.

5
00:00:16,800 --> 00:00:20,000
Try it free this week. Link in bio.

Saved -> session-04_promo.srt


### Example · `session-04_jingle.py`

In [3]:
"""
Session 4 · Example 3 — Make sound from numbers: a 2-second jingle.

Every audio tool, AI or not, ends up writing samples: numbers that describe
air pressure thousands of times per second. Here we synthesise a short
four-note jingle with pure maths (sine waves), add a fade in/out to avoid
clicks, save it as a WAV file with the standard library, and plot the
waveform. AI music tools do the same final step, only their numbers come
from a neural network instead of a sine formula.

Run:  python session-04_jingle.py
Creates: session-04_jingle.wav and session-04_waveform.png (matplotlib)
"""
import math
import struct
import wave

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RATE = 8000                     # samples per second (phone quality, keeps file small)
NOTES = [("C5", 523.25), ("E5", 659.25), ("G5", 783.99), ("C6", 1046.50)]
NOTE_LEN = 0.5                  # seconds per note
FADE = 0.05                     # 50 ms fade at each note edge

samples = []
for name, freq in NOTES:
    n = int(RATE * NOTE_LEN)
    for i in range(n):
        t = i / RATE
        env = min(1.0, t / FADE, (NOTE_LEN - t) / FADE)   # fade in and out
        samples.append(0.6 * env * math.sin(2 * math.pi * freq * t))

with wave.open("session-04_jingle.wav", "wb") as w:
    w.setnchannels(1)           # mono
    w.setsampwidth(2)           # 16-bit
    w.setframerate(RATE)
    w.writeframes(b"".join(struct.pack("<h", int(s * 32767)) for s in samples))

print(f"Notes: {' '.join(n for n, _ in NOTES)}")
print(f"Samples written: {len(samples)} ({len(samples)/RATE:.1f} s at {RATE} Hz)")
print(f"Peak amplitude: {max(abs(s) for s in samples):.2f} (1.0 = clipping)")
print("Saved -> session-04_jingle.wav")

# Plot the whole jingle plus a zoom on the first 10 ms to see the wave shape
fig, (a, b) = plt.subplots(2, 1, figsize=(8, 4))
a.plot([i / RATE for i in range(len(samples))], samples, color="#FF6B1A", linewidth=0.4)
a.set_title("Jingle waveform (2 s): four notes with fades")
a.set_xlabel("seconds")
zoom = int(RATE * 0.01)
b.plot([1000 * i / RATE for i in range(zoom)], samples[:zoom], color="#A78BFA", marker=".")
b.set_title("First 10 ms: a sine wave fading in")
b.set_xlabel("milliseconds")
fig.tight_layout()
fig.savefig("session-04_waveform.png", dpi=80)
print("Saved -> session-04_waveform.png")

Notes: C5 E5 G5 C6
Samples written: 16000 (2.0 s at 8000 Hz)
Peak amplitude: 0.60 (1.0 = clipping)
Saved -> session-04_jingle.wav
Saved -> session-04_waveform.png


### Example · `session-04_project_storyboard.py`

In [4]:
"""
Session 4 · Mini project (full solution) — Storyboard to edit plan.

Real-life problem: the department wants a 45-second promo video made with AI
clips (text-to-video), an AI voiceover and captions. Before generating
anything (each clip costs credits and time), plan it:

  * a shot list with a text-to-video prompt for every shot,
  * narration per shot, timed at 150 wpm,
  * a check that each shot is long enough for its narration,
  * total runtime vs the 45 s target,
  * a JSON edit plan an editor (or a script) can follow.

Run:  python session-04_project_storyboard.py
Creates: session-04_edit_plan.json
"""
import json

WPM, TARGET = 150, 45
shots = [
    # (shot length s, visual prompt for a text-to-video tool, narration, music cue)
    (6, "Slow drone shot over a university campus at sunrise, warm light, cinematic",
        "Every great career starts with one good decision.", "soft piano in"),
    (8, "Close-up of hands typing code on a laptop, screen glow, shallow depth of field",
        "In our computer science labs you build real projects from week one.", "beat starts"),
    (8, "Students laughing around a whiteboard covered in diagrams, handheld camera",
        "You learn with people who push you, help you and celebrate with you.", "beat"),
    (7, "A robot arm sorting coloured blocks, macro lens, studio lighting",
        "Robotics, AI and data science, taught by practising engineers.", "beat"),
    (8, "Graduates throwing caps in the air, slow motion, golden hour",
        "Our graduates work at startups and global companies.", "swell"),
    (6, "Logo reveal on dark background with orange glow particles",
        "Applications are open. Visit our website today.", "final hit"),
]

plan, t, problems = [], 0.0, []
print(f"{'#':>2} {'Len':>4} {'Talk':>5}  Visual prompt")
for i, (length, visual, narration, music) in enumerate(shots, 1):
    talk = len(narration.split()) / WPM * 60
    flag = "" if talk <= length - 0.5 else "  <-- narration too long for shot"
    if flag:
        problems.append(i)
    print(f"{i:2} {length:4} {talk:5.1f}  {visual[:58]}{flag}")
    plan.append({"shot": i, "start": t, "end": t + length, "visual_prompt": visual,
                 "narration": narration, "music": music, "caption": narration})
    t += length

print(f"\nRuntime {t:.0f} s vs target {TARGET} s -> {'OK' if abs(t - TARGET) <= 2 else 'adjust shots'}")
print("All narration fits its shot" if not problems else f"Rewrite narration for shots {problems}")
with open("session-04_edit_plan.json", "w") as f:
    json.dump({"title": "CS Department Promo", "fps": 30, "shots": plan}, f, indent=2)
print("Saved -> session-04_edit_plan.json")
print("\nFirst entry of the edit plan:")
print(json.dumps(plan[0], indent=2))

 #  Len  Talk  Visual prompt
 1    6   3.2  Slow drone shot over a university campus at sunrise, warm 
 2    8   4.8  Close-up of hands typing code on a laptop, screen glow, sh
 3    8   5.2  Students laughing around a whiteboard covered in diagrams,
 4    7   3.6  A robot arm sorting coloured blocks, macro lens, studio li
 5    8   3.2  Graduates throwing caps in the air, slow motion, golden ho
 6    6   2.8  Logo reveal on dark background with orange glow particles

Runtime 43 s vs target 45 s -> OK
All narration fits its shot
Saved -> session-04_edit_plan.json

First entry of the edit plan:
{
  "shot": 1,
  "start": 0.0,
  "end": 6.0,
  "visual_prompt": "Slow drone shot over a university campus at sunrise, warm light, cinematic",
  "narration": "Every great career starts with one good decision.",
  "music": "soft piano in",
  "caption": "Every great career starts with one good decision."
}
